In [46]:
import os
print(os.getcwd())

/home/unimelb.edu.au/rbengtsson/work/library_screen/src


In [1]:
import os
from pathlib import Path
os.chdir("/home/unimelb.edu.au/rbengtsson/work/library_screen/src")
import bedtools_closest as bt
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import utils as utils
import numpy as np

home_dir = Path("/home/unimelb.edu.au/rbengtsson/work/library_screen/")
bed_file1 = home_dir / "data/raw/MT007544.1/PspCas13b_guides.bed"
bed_file2 = home_dir / "data/raw/MT007544.1/RfxCas13d_guides.bed"

### Sort the BED files
sorted_bed1, sorted_bed2 = bt.run_bedtools_srt(bed_file1, bed_file2)


### Filter the first BED file based on guide score
filt_srt_bed2 = bt.filter_guide_score(sorted_bed2, threshold=0.01)
# filt = filt_srt_bed2.filter(
#     lambda x: x[0] == "orf1ab_2").saveas()
# print(filt)

### Get closest coordinates
closest_features = bt.run_bedtools_closest(sorted_bed1, filt_srt_bed2)
# nearby = sorted_bed1.window(filt_srt_bed2, w=50)  # report all B features within 50bp of each A interval


# # # ### Sort the closest features by distance (last column and score)
filtered_results = bt.filter_results(closest_features)


In [111]:
# Separate PspCas13b
genes = filtered_results["gene"].unique()
genes = [g for g in genes if g != "orf1ab_1"]
n = len(genes)

def assign_tracks(df, start_col='qstart', end_col='qend'):
    """
    Assign a track (y-level) to each interval so overlapping guides
    are stacked, non-overlapping guides share a track.
    """
    df = df.sort_values(start_col).reset_index(drop=True)
    track_ends = []  # tracks[i] = end position of last interval placed on that track
    
    tracks = []
    for _, row in df.iterrows():
        placed = False
        for t, end in enumerate(track_ends):
            if row[start_col] >= end:  # no overlap with last item on this track
                track_ends[t] = row[end_col]
                tracks.append(t)
                placed = True
                break
        if not placed:
            track_ends.append(row[end_col])
            tracks.append(len(track_ends) - 1)

    df['track'] = tracks
    return df


In [170]:
# Pre-compute track counts so row heights scale with content
track_counts = []
for gene in genes:
    gene_df = filtered_results[filtered_results["gene"] == gene].reset_index(drop=True)
    pspcas13b = gene_df.iloc[:,0:6].groupby('PspCas13b index').first().reset_index()
    rfxcas13d = (gene_df.iloc[:, np.r_[0, 6:12]]
                 .groupby('RfxCas13d index').first().reset_index())
    psp_n = assign_tracks(pspcas13b.copy(), start_col='qstart', end_col='qend')['track'].nunique()
    rfx_n = assign_tracks(rfxcas13d.copy(), start_col='sstart', end_col='send')['track'].nunique()
    track_counts.append(psp_n + rfx_n + 1)  # +1 for separator gap

total_tracks = sum(track_counts)
row_heights = [tc / total_tracks for tc in track_counts]

In [172]:
gene = "3'UTR"
gene_df = filtered_results[filtered_results["gene"] == gene].reset_index(drop=True)

rfxcas13d = (gene_df.iloc[:, np.r_[0, 6:12]]
                .groupby('RfxCas13d index').first().reset_index())

display(rfxcas13d)

,RfxCas13d index,gene,sstart,send,RfxCas13d guide seq,RfxCas13d target seq,guide score
0,254,3'UTR,246,269,AAGACCAAAAGCTTGCACCATGT,ACATGGTGCAAGCTTTTGGTCTT,0.987711
1,261,3'UTR,785,808,AATTACAACATGATTCCATTCTG,CAGAATGGAATCATGTTGTAATT,0.980053
2,263,3'UTR,247,270,GAAGACCAAAAGCTTGCACCATG,CATGGTGCAAGCTTTTGGTCTTC,0.976930
3,313,3'UTR,248,271,CGAAGACCAAAAGCTTGCACCAT,ATGGTGCAAGCTTTTGGTCTTCG,0.900752
4,325,3'UTR,789,812,CTGTAATTACAACATGATTCCAT,ATGGAATCATGTTGTAATTACAG,0.886076
5,334,3'UTR,249,272,GCGAAGACCAAAAGCTTGCACCA,TGGTGCAAGCTTTTGGTCTTCGC,0.855860
6,21254,3'UTR,246,269,AAGACCAAAAGCTTGCACCATGT,ACATGGTGCAAGCTTTTGGTCTT,0.987711
7,21268,3'UTR,247,270,GAAGACCAAAAGCTTGCACCATG,CATGGTGCAAGCTTTTGGTCTTC,0.976930
8,21330,3'UTR,248,271,CGAAGACCAAAAGCTTGCACCAT,ATGGTGCAAGCTTTTGGTCTTCG,0.900752
9,21345,3'UTR,249,272,GCGAAGACCAAAAGCTTGCACCA,TGGTGCAAGCTTTTGGTCTTCGC,0.855860


In [ ]:
Y_PAD = 2

colors = {"RxfCas13d": "#1f77b4", "PspCas13b": "#ff7f0e"}
legend_added = set()

fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=[f"{gene}" for gene in genes],
    shared_xaxes=False,
    vertical_spacing=0.05,
    row_heights=row_heights,
)

psp_track_ns = {}
rfx_track_ns = {}

for row_idx, gene in enumerate(genes, start=1):

    gene_df = filtered_results[filtered_results["gene"] == gene].reset_index(drop=True)
    pspcas13b = gene_df.iloc[:,0:6].groupby('PspCas13b index').first().reset_index()
    rfxcas13d = (gene_df.iloc[:, np.r_[0, 6:12]]
                 .groupby('RfxCas13d index').first().reset_index())

    # PspCas13b — positive integer tracks (top)
    psp_sub = assign_tracks(pspcas13b, start_col='qstart', end_col='qend')
    psp_track_ns[row_idx] = psp_sub['track'].nunique()

    cas_type = "PspCas13b"
    show_legend = cas_type not in legend_added
    legend_added.add(cas_type)
    for _, row in psp_sub.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['qstart'], row['qend']],
            y=[row['track'], row['track']],
            mode='lines',
            line=dict(width=10, color=colors["PspCas13b"]),
            name=cas_type,
            legendgroup=cas_type,
            showlegend=show_legend,
            hovertemplate=(
                f"<b>PspCas13b</b><br>"
                f"Position: {row['qstart']} - {row['qend']}<br>"
                f"Guide seq: {row['PspCas13b guide seq']}<br>"
                "<extra></extra>"
            ),
        ),
        row=row_idx, col=1)
        show_legend = False  # only first trace shows in legend

    # RfxCas13d — negative integer tracks (bottom)
    rfx_sub = assign_tracks(rfxcas13d, start_col='sstart', end_col='send')
    rfx_track_ns[row_idx] = rfx_sub['track'].nunique()
    cas_type = "RxfCas13d"
    show_legend = cas_type not in legend_added
    legend_added.add(cas_type)
    for _, row in rfx_sub.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['sstart'], row['send']],
            y=[-(row['track'] + 1), -(row['track'] + 1)],
            mode='lines',
            line=dict(width=10, color=colors["RxfCas13d"]),
            name=cas_type,
            legendgroup=cas_type,
            showlegend=show_legend,
            hovertemplate=(
                f"<b>RfxCas13d</b><br>"
                f"Index: {row['RfxCas13d index']}<br>"
                f"Position: {row['sstart']} - {row['send']}<br>"
                f"Guide seq: {row['RfxCas13d guide seq']}<br>"
                f"Target seq: {row['RfxCas13d target seq']}<br>"
                f"Guide score: <b>{row['guide score']:.3f}</b><br>"
                "<extra></extra>"
            ),
        ),
        row=row_idx, col=1)
        show_legend = False  # only first trace shows in legend

    ymax = psp_track_ns[row_idx] - 0.5 + Y_PAD
    ymin = -rfx_track_ns[row_idx] - 0.5 - Y_PAD
    fig.update_yaxes(range=[ymin, ymax], row=row_idx, col=1)

fig.update_xaxes(title_text='Position (nt)', row=n, col=1)
fig.update_yaxes(showticklabels=False, title_text=None)

fig.update_layout(
    title="Guide position overlap by subgenomic region",
    height=320 * n,
    width=900,
    margin=dict(l=30, r=30, t=60, b=20),
    legend=dict(title="Cas type", itemsizing="constant"),
)

fig.show()
